In [1]:

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

In [2]:

from typing import TypedDict, Annotated, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage

class AgentState(TypedDict):
    user_input: str
    route: str
    output: str
    messages: Annotated[list[AnyMessage], add_messages]


In [3]:

from langchain_core.messages import HumanMessage, SystemMessage

VALID_ROUTES = {
    "seo_blog_writer",
    "x_blog_writer",
    "general"
}

In [4]:
def router_node(state: AgentState):
    router_prompt = """
    You are a content routing classifier.

    Classify the user's latest request into exactly one:

    seo_blog_writer:
      Long-form articles, SEO blogs, content marketing.

    x_blog_writer:
      Tweets, X posts, threads, hashtags, short social posts.

    general:
      Greetings, questions, ordinary conversation,
      memory questions, and other requests.

    Return ONLY one of these exact route names.
    Do not explain your answer.
    """

    response = llm.invoke([
        SystemMessage(content=router_prompt),
        HumanMessage(content=state["user_input"])
    ])

    route = response.content.strip().lower()

    if route not in VALID_ROUTES:
        route = "general"

    return {
        "route": route,
        "messages": [
            HumanMessage(content=state["user_input"])
        ]
    }

In [6]:
from langchain_core.tools import tool

@tool
def research_tool(topic: str) -> str:
    """Research a topic in depth for a long-form article.
    Use for background facts, themes, and references.
    This is an expensive tool: use at most twice per request.
    """
    return f"""
    Sample research for: {topic}

    Core themes:
    - Definition and background
    - Real-world applications
    - Benefits and limitations
    - Implementation considerations
    - Future directions

    These are illustrative research notes, not
    verified citations or current factual sources.
    """


@tool
def internet_search_tool(query: str) -> str:
    """Perform a lightweight search for SEO keywords,
    related topics, trends, hashtags, or quick facts.
    """
    return f"""
    Sample search for: {query}

    Suggested keyword phrases:
    - AI agents
    - LangGraph workflow
    - Multi-agent architecture
    - Agentic AI applications

    Suggested hashtags:
    #AI #LangGraph #AgenticAI #MachineLearning

    These are examples, not live search results.
    """

In [7]:
print(research_tool.invoke({
    "topic": "Multi-Agent AI Systems"
}))


    Sample research for: Multi-Agent AI Systems

    Core themes:
    - Definition and background
    - Real-world applications
    - Benefits and limitations
    - Implementation considerations
    - Future directions

    These are illustrative research notes, not
    verified citations or current factual sources.
    


In [8]:
seo_llm = llm.bind_tools(
    [research_tool, internet_search_tool],
    parallel_tool_calls=False
)

SEO_SYSTEM_PROMPT = """
You are a professional SEO blog writer.

Your responsibilities:
1. Understand the user's content request.
2. When research is needed, use research_tool
   for substantive background information.
3. Use research_tool no more than twice.
4. Use internet_search_tool for SEO keywords,
   related topics, and lighter lookups.
5. Avoid redundant tool calls.
6. After obtaining sufficient information,
   write the complete blog post.

Final blog structure:
- SEO-friendly title
- Meta description
- Introduction
- H2 and H3 headings
- Detailed content
- Key takeaways
- Conclusion
- Call to action (CTA)

Use the available conversation context.

Do not invent citations, statistics, or sources.
If tools return sample data, do not represent
the results as verified research.

Finish with the actual blog, not a description
of what you intend to write.
"""

def seo_blog_writer(state: AgentState):
    response = seo_llm.invoke([
        SystemMessage(content=SEO_SYSTEM_PROMPT),
        *state["messages"]
    ])

    update = {
        "messages": [response]
    }

    # Tool-call messages may not contain a final answer.
    if not response.tool_calls:
        update["output"] = response.content

    return update

In [9]:

x_llm = llm.bind_tools(
    [internet_search_tool],
    parallel_tool_calls=False
)

X_SYSTEM_PROMPT = """
You are an expert X/Twitter content writer.

Your responsibilities:
1. Write engaging, concise X posts.
2. Use internet_search_tool when fresh trends,
   hashtags, or facts are needed.
3. Avoid unnecessary search calls.
4. Maintain the user's intended tone.
5. Write a complete, ready-to-publish post.

Rules:
- Final output must be under 280 characters.
- Use concise wording.
- Use relevant emojis when appropriate.
- Use no more than 2 relevant hashtags.
- Do not invent live trends or statistics.
- Do not include explanations or commentary
  outside the final post.
"""

def x_blog_writer(state: AgentState):
    response = x_llm.invoke([
        SystemMessage(content=X_SYSTEM_PROMPT),
        *state["messages"]
    ])

    update = {
        "messages": [response]
    }

    if not response.tool_calls:
        update["output"] = response.content

    return update
